## Fetch Latest AML Model & Deploy to KServe with Triton Inference Server

#### Fetches the latest registered Triton model from the AML model registry and deploys it via KServe — no pipeline retraining required.

### What this notebook does

1. **Fetch** the latest `triton_model` from the AML model registry (no retraining, no pipeline)
2. **Download** the model repository locally to verify the Triton layout and extract metadata
3. **Resolve** the Azure Blob Storage URI for the model so KServe can pull it at runtime
4. **Configure** Kubernetes access to the AKS cluster backing the AML workspace
5. **Deploy** a KServe `InferenceService` using the native Triton Inference Server runtime
6. **Test** the endpoint using the KFServing V2 inference protocol

> **Prerequisites**
> - A `triton_model` already registered in the AML model registry
>   (run `train-on-ai-platform-aks-triton.ipynb` at least once to create one)
> - KServe installed in the target AKS cluster (`kubectl get crds | grep inferenceservice`)
> - GPU nodes with NVIDIA drivers OR `config.pbtxt` configured with `KIND_CPU` (the model
>   created by `train-on-ai-platform-aks-triton.ipynb` already uses `KIND_CPU` for ONNX Runtime)

## Install Required Packages

Adds `kserve`, `kubernetes`, and `azure-storage-blob` on top of the base requirements.
Run once per kernel restart.

In [1]:

%pip install "azure-ai-ml>=1.12.0" \
             "azure-identity>=1.14.0" \
             "scikit-learn>=1.3.0" \
             "onnxruntime>=1.16.0" \
             "numpy>=1.24.0" \
             "requests>=2.31.0" \
             "kubernetes>=28.1.0" \
             "kserve>=0.12.0" \
             "azure-storage-blob>=12.19.0"
# azure-mgmt-storage and azure-mgmt-containerservice may conflict with system packages;
# install to user site-packages to avoid permission errors.
%pip install "azure-mgmt-storage>=21.0.0" \
             "azure-mgmt-containerservice>=28.0.0" \
             --user -q


  Using cached protobuf-7.34.0-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)


  Using cached psutil-5.9.8-cp36-abi3-manylinux_2_12_x86_64.manylinux2010_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (21 kB)


  Using cached protobuf-5.29.6-cp38-abi3-manylinux2014_x86_64.whl.metadata (592 bytes)


Using cached protobuf-5.29.6-cp38-abi3-manylinux2014_x86_64.whl (320 kB)
Using cached psutil-5.9.8-cp36-abi3-manylinux_2_12_x86_64.manylinux2010_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (288 kB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kfp 2.7.0 requires kubernetes<27,>=8.0.0, but you have kubernetes 35.0.0 which is incompatible.
kfp 2.7.0 requires protobuf<5,>=4.21.1, but you have protobuf 5.29.6 which is incompatible.
kfp-pipeline-spec 0.3.0 requires protobuf<5,>=4.21.1, but you have protobuf 5.29.6 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Project Root Setup ────────────────────────────────────────────────────────
# Ensure working directory is always the project root so that all relative paths
# (./src/..., ./artifacts/...) resolve correctly regardless of where Jupyter started.
import os, sys
from pathlib import Path

_nb_file = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
if _nb_file:
    _project_root = str(Path(_nb_file).resolve().parent)
    os.chdir(_project_root)
else:
    _project_root = os.getcwd()

if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

os.makedirs(os.path.join(_project_root, 'artifacts'), exist_ok=True)

print(f'Project root : {_project_root}')
print(f'Working dir  : {os.getcwd()}')

Project root : /home/jovyan/ai-platform-aml-triton-examples
Working dir  : /home/jovyan/ai-platform-aml-triton-examples


| Compute Name | Time Slicing Options     | Base Node Guaranteed CPU | Base Node Guaranteed Memory |
|--------------|------------------|---------------|------------------|
| cpu          | -2, -4           | 14            | 46Gi             |
| gput41       | -2, -4           | 6             | 44Gi             |
| gpuv1001     | -2, -4           | 4             | 98Gi             |
| gpuv1002     | -2, -4           | 10            | 206Gi            |
| gpua100      | -2, -4           | 22            | 202Gi            |
| gput44       | -2, -4, -8       | 56            | 410Gi            |
| gpuv1004     | -2, -4, -8       | 22            | 422Gi            |

## GPU Time slicing (Short for TS)

Allows up to 8 workloads onto one GPU node to run everything in concurrency, occupying one Node, distributing execution time evenly every ~3 seconds. This way, once auto-scaled, GPU node is available to serve new workloads almost instantly, or, host multiple deployments at the same time.

To utilise time slicing, append 2, 4 or 8 in the compute type:

Example: __gput44-2__

Another example, if you wish to share A100 node between 4 workloads, specify this: __gpuv1002-4__
This way, it will auto select right Node type to be shared for 4 different workloads: jobs/pipelines/deploys equally distributing resources.

For regular, non time slicing cases, simply indicate compute name without any suffix, e.g.: __gpua1001__

## Configuration

Edit the variables below before running:
- `model_name_to_fetch` — exact AML model name **or** a name prefix (e.g. `"triton-iris-fil"`); the notebook auto-selects the most recently registered match
- `aks_compute_name` — AML compute target whose underlying AKS cluster will host the KServe `InferenceService`
- `kserve_namespace` — Kubernetes namespace where KServe CRDs are installed

In [3]:

# ── Azure ML environments ──────────────────────────────────────────────────────
triton_environment = "tritonNcdEnv:23.08.02-py3"   # Triton Inference Server environment

# ── Model to fetch from AML model registry ────────────────────────────────────
# Set to the EXACT registered model name, e.g. "triton-iris-fil-20240301120000",
# OR to a name prefix to auto-select the most recently created matching model.
model_name_to_fetch = "triton-iris-fil"   # prefix — adjust if your model uses a different name
model_version       = None                 # None → fetch latest version

# When set, the fetch function skips any registered model whose Triton repository
# does not contain a subdirectory with this name (e.g. "iris_classifier").
# Set to None to accept any model.
required_triton_model_name = "iris_classifier"

# ── KServe deployment settings ────────────────────────────────────────────────
kserve_namespace       = "unified"              # K8s namespace where KServe CRDs are installed
inference_service_name = "iris-triton"           # Name for the KServe InferenceService resource
k8s_secret_name        = "azure-storage-secret" # K8s Secret that holds Azure Blob credentials

# ── AKS cluster (to obtain kubeconfig when running outside the cluster) ───────
# Only used as fallback when in-cluster config is not available.
aks_compute_name = "cpu-2"

# ── Resource requests / limits for the Triton predictor pod ───────────────────
request_cpu = "500m"
request_ram = "2Gi"
limit_cpu   = "2"
limit_ram   = "4Gi"

tags = {
    "Purpose":   "Project Resources",
    "by_person": "by_person",   # recommended: replace with your name
}


## Import libraries

In [4]:

import os, re, json, subprocess, datetime, time
import glob as glob_mod
import shutil
from pathlib import Path

# azure-mgmt-* packages are installed --user to avoid system site-packages conflicts;
# ensure the user site-packages directory is on sys.path before importing them.
import sys
_user_site = os.path.expanduser("~/.local/lib/python3.11/site-packages")
if _user_site not in sys.path:
    sys.path.insert(0, _user_site)

from azure.ai.ml import MLClient
from azure.identity import ManagedIdentityCredential
from azure.ai.ml.entities import Model
from azure.storage.blob import BlobServiceClient

from kubernetes import client as k8s_client, config as k8s_config
from kserve import (
    KServeClient,
    V1beta1InferenceService,
    V1beta1InferenceServiceSpec,
    V1beta1PredictorSpec,
    V1beta1TritonSpec,
    constants,
)

# Service functions
instanceType           = "defaultinstancetype"
nb_prefix              = os.getenv("NB_PREFIX", "/notebook/unknown/unknown")
_, namespace, pod_name = nb_prefix.strip("/").split("/")
AZURE_CLIENT_ID        = os.getenv('AZURE_CLIENT_ID')
credential             = ManagedIdentityCredential(client_id=os.getenv("AZURE_CLIENT_ID"))


## Import Helper script:

In [5]:
from src._helpers.load_tags import load_tags
tags = load_tags(tags={}, namespace=namespace, pod_name=pod_name)
for k, v in tags.items():
    globals()[k] = v

print(tags)

{'CostAllocationCode': 'C.ITD.07.017', 'CostAllocationType': 'WBS', 'Project': 'unified', 'SubProject': 'unified', 'WBS': 'C.ITD.07.017', 'aks_cluster': 'dev-aurora-test-01', 'aml_location': 'northeurope', 'aml_workspace': 'unified-amlws-dev', 'aml_workspace_rg': 'unified-rg-dev', 'subscription_id': '019958ea-fe2c-4e14-bbd9-0d2db8ed7cfc', 'submitted_by': 'unified', 'notebook_pod': 'test-pshah-medium'}


## Build ML client

In [6]:
subscription_id = subscription_id
resource_group  = aml_workspace_rg
workspace       = aml_workspace

ml_client = MLClient(credential, subscription_id, resource_group, workspace)
print(f"ML Client workspace: {ml_client.workspace_name}")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


ML Client workspace: unified-amlws-dev


## Fetch Latest Triton Model from AML Registry

Instead of retraining, we look up the most recently registered model whose name matches
`model_name_to_fetch` (exact name or prefix search). The model must already exist as a
`triton_model` type — run `train-on-ai-platform-aks-triton.ipynb` at least once if it does not.

The registered model must contain a valid Triton model repository layout:
```
<model_name>/
  config.pbtxt
  1/
    model.onnx
```

In [7]:

def _blob_uri_parts(aml_model_path):
    """Parse an azureml:// model path into (datastore_name, blob_prefix)."""
    m = re.search(r'/datastores/([^/]+)/paths/(.+?)/?$', aml_model_path)
    if not m:
        return None, None
    return m.group(1), m.group(2).rstrip('/')


def _model_has_triton_subdir(ml_client, credential, aml_model, required_triton_model_name):
    """
    Return True if the AML model's blob storage path contains
    <required_triton_model_name>/config.pbtxt without downloading the whole model.
    Falls back to True (optimistic) if the blob check cannot be performed.
    """
    ds_name, blob_prefix = _blob_uri_parts(aml_model.path)
    if not ds_name:
        return True  # cannot check, assume OK

    try:
        ds              = ml_client.datastores.get(ds_name)
        blob_service    = BlobServiceClient(
            account_url=f"https://{ds.account_name}.blob.core.windows.net",
            credential=credential,
        )
        container_client = blob_service.get_container_client(ds.container_name)
        target_blob      = f"{blob_prefix}/{required_triton_model_name}/config.pbtxt"
        blob_client      = container_client.get_blob_client(target_blob)
        blob_client.get_blob_properties()   # raises if not found
        return True
    except Exception:
        return False


def fetch_latest_aml_model(ml_client, credential, name_or_prefix,
                            required_triton_model_name=None, version=None):
    """
    Fetch a model from the AML registry by exact name or name prefix.

    If required_triton_model_name is set, the function skips any candidate
    whose Triton model repository does not contain that model subdirectory,
    using a fast blob-existence check (no full download needed).

    Strategy:
      1. Exact name match → return immediately (no blob check when version given).
      2. Prefix search sorted newest-first → return first that passes the blob check.
    """
    # Strategy 1: exact name
    try:
        if version:
            m = ml_client.models.get(name=name_or_prefix, version=str(version))
        else:
            m = ml_client.models.get(name=name_or_prefix, label="latest")
        print(f"Exact match — model '{m.name}' version {m.version}")
        return m
    except Exception as e1:
        print(f"Exact name lookup failed ({type(e1).__name__}), searching by prefix...")

    # Strategy 2: prefix search sorted newest-first
    candidates = sorted(
        [m for m in ml_client.models.list() if m.name.startswith(name_or_prefix)],
        key=lambda m: (
            m.creation_context.created_at
            if m.creation_context and m.creation_context.created_at
            else datetime.datetime.min.replace(tzinfo=datetime.timezone.utc)
        ),
        reverse=True,
    )
    if not candidates:
        raise RuntimeError(
            f"No models found with name or prefix '{name_or_prefix}' "
            f"in workspace '{ml_client.workspace_name}'.\n"
            "Run train-on-ai-platform-aks-triton.ipynb first to register a triton_model."
        )

    print(f"Found {len(candidates)} candidate(s):")
    for c in candidates[:5]:
        ts = c.creation_context.created_at if c.creation_context else "unknown"
        print(f"  {c.name}  v{c.version}  created={ts}")

    for candidate in candidates:
        m = ml_client.models.get(name=candidate.name, label="latest")
        if required_triton_model_name:
            has_it = _model_has_triton_subdir(ml_client, credential, m, required_triton_model_name)
            if not has_it:
                print(f"  Skipping '{m.name}' — does not contain '{required_triton_model_name}/'")
                continue
            print(f"  '{m.name}' contains '{required_triton_model_name}/' ✓")
        print(f"\nSelected: '{m.name}' version {m.version}")
        return m

    raise RuntimeError(
        f"None of the {len(candidates)} candidate model(s) with prefix '{name_or_prefix}' "
        f"contain a '{required_triton_model_name}/config.pbtxt' in their Triton repository.\n"
        "Register a model that includes the required Triton model subdirectory."
    )


aml_model = fetch_latest_aml_model(
    ml_client, credential,
    model_name_to_fetch,
    required_triton_model_name=required_triton_model_name,
    version=model_version,
)

print(f"\nModel details:")
print(f"  Name        : {aml_model.name}")
print(f"  Version     : {aml_model.version}")
print(f"  Type        : {aml_model.type}")
print(f"  Path (AML)  : {aml_model.path}")
print(f"  Description : {aml_model.description}")


Exact name lookup failed (ResourceNotFoundError), searching by prefix...


Found 10 candidate(s):
  triton-iris-fil-20260303202754  vNone  created=2026-03-03 20:27:57.207540+00:00
  triton-iris-fil-20260302200824  vNone  created=2026-03-02 20:08:27.121973+00:00
  triton-iris-fil-20260302194954  vNone  created=2026-03-02 19:49:56.277827+00:00
  triton-iris-fil-20260228014420  vNone  created=2026-02-28 01:44:22.003156+00:00
  triton-iris-fil-20260227235839  vNone  created=2026-02-27 23:58:40.694255+00:00


  'triton-iris-fil-20260303202754' contains 'iris_classifier/' ✓

Selected: 'triton-iris-fil-20260303202754' version 1

Model details:
  Name        : triton-iris-fil-20260303202754
  Version     : 1
  Type        : triton_model
  Path (AML)  : azureml://subscriptions/019958ea-fe2c-4e14-bbd9-0d2db8ed7cfc/resourceGroups/unified-rg-dev/workspaces/unified-amlws-dev/datastores/workspaceblobstore/paths/LocalUpload/74f7bb2ac94eee02a4e0bc7bbcfbe99126533adf7cd297dbc439c3ec8d30cc67/triton_model_repo
  Description : Two-model Triton repository: iris_classifier (sklearn RFC) + pytorch_sine (MLP sine regressor). KFServing V2 API. CPU inference.


## Download Model Repository Locally

We download the registered model to:
- Verify it has a valid Triton repository layout (`config.pbtxt`, `model.onnx`)
- Extract `triton_model_name` and `n_features` for the test payload

The actual serving uses the blob storage URI (next cell) — this download is for local inspection only.

In [8]:

download_dir = os.path.join(_project_root, "artifacts", "fetched_model")
if os.path.exists(download_dir):
    shutil.rmtree(download_dir)
os.makedirs(download_dir, exist_ok=True)

print(f"Downloading model '{aml_model.name}' v{aml_model.version} ...")
ml_client.models.download(
    name=aml_model.name,
    version=aml_model.version,
    download_path=download_dir,
)
print(f"Downloaded to: {download_dir}")

# ── Locate Triton model repository root ───────────────────────────────────────
# config.pbtxt lives inside <triton_model_name>/, so parent.parent is the repo root.
config_files = list(Path(download_dir).rglob("config.pbtxt"))
if not config_files:
    raise FileNotFoundError(
        f"No config.pbtxt found under {download_dir}. "
        "Ensure the registered model is a Triton model repository (type=triton_model)."
    )

# If the repository contains multiple models, prefer required_triton_model_name.
# Fall back to the first config found if the required one is absent.
if required_triton_model_name:
    preferred = [c for c in config_files if c.parent.name == required_triton_model_name]
    config_path = preferred[0] if preferred else config_files[0]
else:
    config_path = config_files[0]

triton_model_dir  = config_path.parent          # e.g. .../iris_classifier/
local_model_repo  = triton_model_dir.parent     # e.g. .../triton_model_repo/
triton_model_name = triton_model_dir.name       # e.g. "iris_classifier"

print(f"\nTriton model repository : {local_model_repo}")
print(f"Model name (config)     : {triton_model_name}")
print(f"config.pbtxt path       : {config_path}")

# Verify expected artefacts
onnx_files = list(triton_model_dir.rglob("model.onnx"))
pkl_files  = list(triton_model_dir.rglob("model.pkl"))
print(f"model.onnx : {[str(p.relative_to(local_model_repo)) for p in onnx_files]}")
print(f"model.pkl  : {[str(p.relative_to(local_model_repo)) for p in pkl_files]}")

if not onnx_files:
    raise FileNotFoundError(
        f"model.onnx not found under {triton_model_dir}. "
        "Re-register the model using train-on-ai-platform-aks-triton.ipynb "
        "which performs the sklearn→ONNX conversion before registration."
    )

# ── Extract n_features from config.pbtxt (used for the test payload) ──────────
config_txt = config_path.read_text()
_dims = re.findall(r'dims:\s*\[(\d+)\]', config_txt)
n_features = int(_dims[0]) if _dims else 4    # fallback: Iris has 4 features
n_classes  = 3                                 # Iris: setosa / versicolor / virginica

print(f"\nn_features (from config): {n_features}")
print(f"\nconfig.pbtxt:\n{config_txt}")


Downloaded to: /home/jovyan/ai-platform-aml-triton-examples/artifacts/fetched_model

Triton model repository : /home/jovyan/ai-platform-aml-triton-examples/artifacts/fetched_model/triton-iris-fil-20260303202754/triton_model_repo
Model name (config)     : iris_classifier
config.pbtxt path       : /home/jovyan/ai-platform-aml-triton-examples/artifacts/fetched_model/triton-iris-fil-20260303202754/triton_model_repo/iris_classifier/config.pbtxt
model.onnx : ['iris_classifier/1/model.onnx']
model.pkl  : ['iris_classifier/1/model.pkl']

n_features (from config): 4

config.pbtxt:
name: "iris_classifier"
backend: "onnxruntime"
max_batch_size: 0

input [
  {
    name: "float_input"
    data_type: TYPE_FP32
    dims: [-1, 4]
  }
]

output [
  {
    name: "label"
    data_type: TYPE_INT64
    dims: [-1]
  },
  {
    name: "probabilities"
    data_type: TYPE_FP32
    dims: [-1, 3]
  }
]

instance_group [
  {
    kind: KIND_CPU
    count: 1
  }
]



## Resolve Azure Blob Storage URI for the Model

AML stores registered models in its default datastore (Azure Blob Storage). The `model.path`
attribute is an `azureml://` URI. We convert it to an `https://` blob URI that KServe's
storage initializer can download from at pod startup.

The `storageUri` must point to the **Triton model repository root** — the directory that
*contains* the model-name subdirectory (e.g. the directory containing `iris_classifier/`).

In [9]:
model_aml_path = aml_model.path
print(f"AML model path : {model_aml_path}")

# Parse datastore name and blob prefix from the azureml:// URI.
# Format: azureml://datastores/<name>/paths/<blob_prefix>/
#     OR: azureml://subscriptions/.../datastores/<name>/paths/<blob_prefix>/
_match = re.search(r'/datastores/([^/]+)/paths/(.+?)/?$', model_aml_path)
if not _match:
    raise ValueError(
        f"Cannot parse AML model path: {model_aml_path}\n"
        "Expected format: azureml://datastores/<name>/paths/<prefix>/"
    )

datastore_name = _match.group(1)
blob_path      = _match.group(2).rstrip('/')
print(f"Datastore name : {datastore_name}")
print(f"Blob prefix    : {blob_path}")

# Look up the datastore to get storage account name and container name
datastore       = ml_client.datastores.get(datastore_name)
storage_account = datastore.account_name
container_name  = datastore.container_name
print(f"Storage account: {storage_account}")
print(f"Container      : {container_name}")

# Full HTTPS URI — KServe storage initializer downloads everything under this prefix.
# This is the model REPOSITORY ROOT (the directory containing <triton_model_name>/).
blob_storage_uri = (
    f"https://{storage_account}.blob.core.windows.net/{container_name}/{blob_path}"
)
print(f"\nBlob storage URI (KServe storageUri):\n  {blob_storage_uri}")

AML model path : azureml://subscriptions/019958ea-fe2c-4e14-bbd9-0d2db8ed7cfc/resourceGroups/unified-rg-dev/workspaces/unified-amlws-dev/datastores/workspaceblobstore/paths/LocalUpload/74f7bb2ac94eee02a4e0bc7bbcfbe99126533adf7cd297dbc439c3ec8d30cc67/triton_model_repo
Datastore name : workspaceblobstore
Blob prefix    : LocalUpload/74f7bb2ac94eee02a4e0bc7bbcfbe99126533adf7cd297dbc439c3ec8d30cc67/triton_model_repo


Storage account: unifiedamlsadev
Container      : azureml-blobstore-36930183-c319-42bf-a68d-9e2b019e627c

Blob storage URI (KServe storageUri):
  https://unifiedamlsadev.blob.core.windows.net/azureml-blobstore-36930183-c319-42bf-a68d-9e2b019e627c/LocalUpload/74f7bb2ac94eee02a4e0bc7bbcfbe99126533adf7cd297dbc439c3ec8d30cc67/triton_model_repo


## Obtain Storage Credentials for KServe

KServe's storage initializer needs credentials to download the model from Azure Blob Storage.

**Options (most to least preferred):**
1. **Workload Identity / Managed Identity** — zero-secret; requires cluster-level AAD pod identity
2. **Storage Account Key** — straightforward; works everywhere
3. **SAS Token** — time-limited, least-privilege

This notebook tries to retrieve the account key from the AML datastore credentials object.
If the datastore uses identity-based access (no stored key), it falls back to the notebook
pod's managed identity (`AZURE_CLIENT_ID`).

In [10]:

# ── Retrieve Storage Account Key ──────────────────────────────────────────────
# KServe storage initializer pods run without the notebook pod's AAD pod identity,
# so they cannot use managed identity to access blob storage. We retrieve the
# storage account key using our notebook's managed identity credential (which does
# have the identity) and inject it into the K8s secret as AZURE_STORAGE_ACCESS_KEY.
#
# This requires the managed identity to have the
# "Microsoft.Storage/storageAccounts/listkeys/action" permission (e.g. via
# "Storage Account Contributor" or a custom role on the storage account).

storage_account_key = None

# Method 1: check AML datastore credentials object
try:
    ds_creds = datastore.credentials
    if hasattr(ds_creds, 'account_key') and ds_creds.account_key:
        storage_account_key = ds_creds.account_key
        print("Storage key retrieved from AML datastore credentials.")
except Exception:
    pass

# Method 2: Azure Storage Management API (works when method 1 returns nothing)
if not storage_account_key:
    try:
        from azure.mgmt.storage import StorageManagementClient
        storage_mgmt = StorageManagementClient(credential, subscription_id)
        keys = storage_mgmt.storage_accounts.list_keys(resource_group, storage_account)
        storage_account_key = keys.keys[0].value
        print("Storage key retrieved via Azure Storage Management API.")
    except Exception as e:
        print(f"Could not retrieve storage key via Management API: {e}")

if storage_account_key:
    print(f"Storage account key obtained (length={len(storage_account_key)}).")
else:
    print(
        "WARNING: Could not obtain a storage account key.\n"
        "KServe storage initializer pods will attempt managed identity auth.\n"
        "If the InferenceService fails to become Ready, check storage initializer logs."
    )


Storage key retrieved via Azure Storage Management API.
Storage account key obtained (length=88).


## Patch config.pbtxt and Re-upload to Blob

The registered model's `config.pbtxt` was generated with `max_batch_size: 64` and `dims: [1]`
for the `label` output. With `max_batch_size > 0`, Triton prepends a batch dimension and
expects `label` shape `[-1, 1]` (2D), but `skl2onnx` exports `label` as 1D shape `[-1]`.

This cell fixes the config to use `max_batch_size: 0` (explicit batch dim in each tensor shape)
and uploads the patched file directly to Azure Blob Storage, replacing the registered model's
copy. The blob URI remains the same, so the next deployment picks up the fixed config.

In [11]:

# ── Fixed config.pbtxt ────────────────────────────────────────────────────────
# max_batch_size: 0  →  Triton does NOT prepend an implicit batch dimension.
#   All tensor shapes must be declared fully (including the batch axis).
#   skl2onnx exports:
#     float_input  → shape [N, 4]    → dims: [-1, 4]
#     label        → shape [N]       → dims: [-1]
#     probabilities→ shape [N, 3]    → dims: [-1, 3]
#
# This matches what the ONNX model actually produces and what the KFServing V2
# scoring cell sends (shape [1, 4] for a single-sample request).

fixed_config = f"""name: "{triton_model_name}"
backend: "onnxruntime"
max_batch_size: 0

input [
  {{
    name: "float_input"
    data_type: TYPE_FP32
    dims: [-1, {n_features}]
  }}
]

output [
  {{
    name: "label"
    data_type: TYPE_INT64
    dims: [-1]
  }},
  {{
    name: "probabilities"
    data_type: TYPE_FP32
    dims: [-1, {n_classes}]
  }}
]

instance_group [
  {{
    kind: KIND_CPU
    count: 1
  }}
]
"""

print("Fixed config.pbtxt:")
print(fixed_config)

# ── Write patched config locally ──────────────────────────────────────────────
local_config_path = config_path   # set by cell 18 (the downloaded config.pbtxt path)
local_config_path.write_text(fixed_config)
print(f"Wrote patched config to: {local_config_path}")

# ── Upload patched config to Azure Blob Storage ───────────────────────────────
# blob_path and container_name were resolved in cell 20.
# The config.pbtxt blob key = <blob_path>/<triton_model_name>/config.pbtxt
if storage_account_key:
    from azure.storage.blob import BlobServiceClient as _BSC
    _blob_svc = _BSC(
        account_url=f"https://{storage_account}.blob.core.windows.net",
        credential=storage_account_key,
    )
    _container = _blob_svc.get_container_client(container_name)
    config_blob_key = f"{blob_path}/{triton_model_name}/config.pbtxt"
    with open(local_config_path, "rb") as _f:
        _container.upload_blob(
            name=config_blob_key,
            data=_f,
            overwrite=True,
        )
    print(f"Uploaded patched config.pbtxt to blob: {config_blob_key}")
else:
    print(
        "WARNING: No storage account key — cannot upload patched config.pbtxt.\n"
        "Triton will use the original (broken) config and iris_classifier will fail to load."
    )


Fixed config.pbtxt:
name: "iris_classifier"
backend: "onnxruntime"
max_batch_size: 0

input [
  {
    name: "float_input"
    data_type: TYPE_FP32
    dims: [-1, 4]
  }
]

output [
  {
    name: "label"
    data_type: TYPE_INT64
    dims: [-1]
  },
  {
    name: "probabilities"
    data_type: TYPE_FP32
    dims: [-1, 3]
  }
]

instance_group [
  {
    kind: KIND_CPU
    count: 1
  }
]

Wrote patched config to: /home/jovyan/ai-platform-aml-triton-examples/artifacts/fetched_model/triton-iris-fil-20260303202754/triton_model_repo/iris_classifier/config.pbtxt


Uploaded patched config.pbtxt to blob: LocalUpload/74f7bb2ac94eee02a4e0bc7bbcfbe99126533adf7cd297dbc439c3ec8d30cc67/triton_model_repo/iris_classifier/config.pbtxt


## Configure Kubernetes Client (AKS Credentials)

We resolve the AKS cluster name and resource group from the AML compute target resource ID,
then call `az aks get-credentials` to merge the cluster's kubeconfig.

In [12]:

# ── Configure Kubernetes client ───────────────────────────────────────────────
# When running inside the AKS cluster (e.g. from a JupyterHub notebook pod),
# in-cluster config is available and preferred — no az CLI needed.
# When running from outside the cluster, fall back to az aks get-credentials.

try:
    k8s_config.load_incluster_config()
    print("Kubernetes client configured via in-cluster config.")
    _using_incluster = True
except k8s_config.ConfigException:
    _using_incluster = False
    print("Not running inside AKS — falling back to az aks get-credentials...")

    # Resolve AKS cluster name + resource group from the AML compute target
    compute     = ml_client.compute.get(aks_compute_name)
    resource_id = compute.resource_id
    parts              = resource_id.split('/')
    aks_resource_group = parts[parts.index('resourceGroups') + 1]
    aks_cluster_name   = parts[-1]
    print(f"AKS cluster       : {aks_cluster_name}")
    print(f"AKS resource group: {aks_resource_group}")

    result = subprocess.run(
        [
            'az', 'aks', 'get-credentials',
            '--resource-group', aks_resource_group,
            '--name',           aks_cluster_name,
            '--overwrite-existing',
        ],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"stderr: {result.stderr}")
        raise RuntimeError(
            "az aks get-credentials failed. "
            "Ensure the 'az' CLI is installed and you are authenticated."
        )
    print(result.stdout.strip() or "kubeconfig updated.")
    k8s_config.load_kube_config()

# Quick connectivity check: list pods in the target namespace (no cluster-scope RBAC needed)
v1   = k8s_client.CoreV1Api()
pods = v1.list_namespaced_pod(namespace=kserve_namespace)
print(f"Connected to cluster. Pods in '{kserve_namespace}': {len(pods.items)}")


Kubernetes client configured via in-cluster config.


Connected to cluster. Pods in 'unified': 7


## Create Kubernetes Secret for Azure Blob Storage Access

KServe's storage initializer reads credentials from a K8s Secret annotated with
`serving.kserve.io/*` annotations. For Azure Blob Storage (`https://` URI) it looks for:

- `AZURE_STORAGE_ACCOUNT`    — storage account name
- `AZURE_STORAGE_ACCESS_KEY` — account key (key-based auth)
- `AZURE_CLIENT_ID`          — managed identity client ID (identity-based auth)

The secret is bound to the `default` ServiceAccount so the storage initializer init-container
inherits credentials automatically.

In [13]:

core_api = k8s_client.CoreV1Api()

# Build secret data with all env vars the KServe storage initializer requires for Azure Blob.
# Required keys (enforced by the ClusterStorageContainer for Azure in this cluster):
#   AZURE_STORAGE_ACCOUNT  — storage account name
#   AZURE_SUBSCRIPTION_ID  — Azure subscription ID
#   AZURE_TENANT_ID        — Azure AD tenant ID
# Auth (one of):
#   AZURE_STORAGE_ACCESS_KEY — account key  (key-based)
#   AZURE_CLIENT_ID          — managed identity / federated token client ID (identity-based)
secret_data = {
    "AZURE_STORAGE_ACCOUNT": storage_account,
    "AZURE_SUBSCRIPTION_ID": subscription_id,
    "AZURE_TENANT_ID":       os.environ.get("AZURE_TENANT_ID", ""),
}
if storage_account_key:
    secret_data["AZURE_STORAGE_ACCESS_KEY"] = storage_account_key
elif AZURE_CLIENT_ID:
    secret_data["AZURE_CLIENT_ID"] = AZURE_CLIENT_ID

secret = k8s_client.V1Secret(
    api_version="v1",
    kind="Secret",
    metadata=k8s_client.V1ObjectMeta(
        name=k8s_secret_name,
        namespace=kserve_namespace,
        annotations={
            "serving.kserve.io/s3-endpoint":          f"{storage_account}.blob.core.windows.net",
            "serving.kserve.io/s3-usehttps":          "1",
            "serving.kserve.io/s3-useanoncredential": "false",
        },
    ),
    string_data=secret_data,
    type="Opaque",
)

# Replace existing secret if present
try:
    core_api.delete_namespaced_secret(name=k8s_secret_name, namespace=kserve_namespace)
    print(f"Replaced existing secret '{k8s_secret_name}'.")
except k8s_client.exceptions.ApiException:
    pass  # did not exist yet

core_api.create_namespaced_secret(namespace=kserve_namespace, body=secret)
print(f"Secret '{k8s_secret_name}' created with keys: {list(secret_data.keys())}")

# ── Bind secret to the default ServiceAccount ─────────────────────────────────
# Allows the KServe storage initializer init-container to inherit credentials.
try:
    sa = core_api.read_namespaced_service_account(name="default", namespace=kserve_namespace)
    bound = [s.name for s in (sa.secrets or [])]
    if k8s_secret_name not in bound:
        sa.secrets = (sa.secrets or []) + [k8s_client.V1ObjectReference(name=k8s_secret_name)]
        core_api.replace_namespaced_service_account(
            name="default", namespace=kserve_namespace, body=sa
        )
        print("Secret bound to 'default' ServiceAccount.")
    else:
        print("Secret already bound to 'default' ServiceAccount.")
except Exception as sa_err:
    print(
        f"Note: could not bind secret to ServiceAccount: {sa_err}\n"
        "Configure service account credential injection manually if model download fails."
    )

# ── Patch mlpipeline-minio-artifact secret ────────────────────────────────────
# The ClusterStorageContainer for Azure Blob in this cluster injects:
#   AZURE_STORAGE_ACCESS_KEY  ← secret 'mlpipeline-minio-artifact' key='secretkey'
# into every storage-initializer init-container. The 'secretkey' field normally holds
# the MinIO artifact store password (not a valid Azure key), causing MAC signature errors.
# We overwrite 'secretkey' with the real Azure storage account key so the storage
# initializer can authenticate against Azure Blob Storage.
#
# The original MinIO key is saved and restored in the Cleanup cell.
if storage_account_key:
    _minio_secret_name = "mlpipeline-minio-artifact"
    try:
        minio_secret = core_api.read_namespaced_secret(
            name=_minio_secret_name, namespace=kserve_namespace
        )
        import base64
        raw_secretkey = minio_secret.data.get("secretkey", b"") if minio_secret.data else b""
        if isinstance(raw_secretkey, bytes):
            _original_minio_secretkey = base64.b64decode(raw_secretkey).decode()
        else:
            _original_minio_secretkey = raw_secretkey
        print(f"Saved original mlpipeline-minio-artifact.secretkey (len={len(_original_minio_secretkey)})")

        # Patch: set secretkey to Azure storage account key
        core_api.patch_namespaced_secret(
            name=_minio_secret_name,
            namespace=kserve_namespace,
            body={"stringData": {"secretkey": storage_account_key}},
        )
        print(
            f"Patched '{_minio_secret_name}.secretkey' with Azure storage account key "
            f"(len={len(storage_account_key)}).\n"
            "Run the Cleanup cell to restore the original value after deployment."
        )
    except k8s_client.exceptions.ApiException as patch_err:
        print(
            f"WARNING: Could not patch '{_minio_secret_name}': {patch_err}\n"
            "Storage initializer may fail to authenticate against Azure Blob Storage."
        )
else:
    print(
        "WARNING: No storage account key available — "
        "skipping mlpipeline-minio-artifact patch.\n"
        "Storage initializer will use managed identity (may fail if identity not assigned)."
    )


Replaced existing secret 'azure-storage-secret'.
Secret 'azure-storage-secret' created with keys: ['AZURE_STORAGE_ACCOUNT', 'AZURE_SUBSCRIPTION_ID', 'AZURE_TENANT_ID', 'AZURE_STORAGE_ACCESS_KEY']
Secret already bound to 'default' ServiceAccount.
Saved original mlpipeline-minio-artifact.secretkey (len=120)
Patched 'mlpipeline-minio-artifact.secretkey' with Azure storage account key (len=88).
Run the Cleanup cell to restore the original value after deployment.


## Deploy KServe InferenceService with Triton Runtime

The `InferenceService` uses `storageUri` to point KServe's storage initializer at the
Azure Blob Storage path where the Triton model repository is stored. At pod startup,
the init-container downloads the repository into `/mnt/models` before Triton starts.

**Expected Triton repository layout at `blob_storage_uri`:**
```
<blob_storage_uri>/
  iris_classifier/
    config.pbtxt
    1/
      model.onnx
```

> **GPU vs CPU:** The standard `nvcr/nvidia/tritonserver` image requires NVIDIA GPU drivers.
> The model registered by `train-on-ai-platform-aks-triton.ipynb` uses `KIND_CPU` in
> `config.pbtxt`, so the ONNX Runtime backend runs on CPU-only nodes without CUDA.

In [14]:

isvc = V1beta1InferenceService(
    api_version=constants.KSERVE_V1BETA1,
    kind=constants.KSERVE_KIND_INFERENCESERVICE,
    metadata=k8s_client.V1ObjectMeta(
        name=inference_service_name,
        namespace=kserve_namespace,
        annotations={
            # Disable Istio sidecar injection if the cluster does not use Istio
            "sidecar.istio.io/inject": "false",
        },
    ),
    spec=V1beta1InferenceServiceSpec(
        predictor=V1beta1PredictorSpec(
            triton=V1beta1TritonSpec(
                # storageUri → Triton model REPOSITORY root
                # (the directory that CONTAINS the model-name subdirectory)
                storage_uri=blob_storage_uri,
                resources=k8s_client.V1ResourceRequirements(
                    requests={"cpu": request_cpu, "memory": request_ram},
                    limits={"cpu": limit_cpu,   "memory": limit_ram},
                ),
            )
        )
    ),
)

ks_client = KServeClient()

# Delete existing InferenceService if it exists (clean redeploy)
try:
    ks_client.delete(name=inference_service_name, namespace=kserve_namespace)
    print(f"Deleted existing InferenceService '{inference_service_name}'. Waiting for removal...")
    time.sleep(10)
except Exception:
    pass  # did not exist yet

ks_client.create(isvc)
print(f"InferenceService '{inference_service_name}' created in namespace '{kserve_namespace}'.")
print("Waiting for the service to become Ready...")


Deleted existing InferenceService 'iris-triton'. Waiting for removal...


InferenceService 'iris-triton' created in namespace 'unified'.
Waiting for the service to become Ready...


## Wait for InferenceService to Reach Ready State

In [15]:
_ISVC_TIMEOUT_SEC = 600   # 10 min — allows time for image pull + model download from blob
_POLL_INTERVAL    = 15

start = time.time()
ready = False

while time.time() - start < _ISVC_TIMEOUT_SEC:
    try:
        status_obj  = ks_client.get(name=inference_service_name, namespace=kserve_namespace)
        status_dict = status_obj if isinstance(status_obj, dict) else status_obj.to_dict()
        conditions  = status_dict.get("status", {}).get("conditions", []) or []

        ready_cond = next((c for c in conditions if c.get("type") == "Ready"), None)
        if ready_cond and ready_cond.get("status") == "True":
            ready = True
            break

        msg = ready_cond.get("message", "waiting") if ready_cond else "initializing"
        print(f"[{int(time.time() - start):4d}s] {msg}")

    except Exception as poll_err:
        print(f"[{int(time.time() - start):4d}s] Polling error: {poll_err}")

    time.sleep(_POLL_INTERVAL)

if not ready:
    raise TimeoutError(
        f"InferenceService '{inference_service_name}' did not become Ready within "
        f"{_ISVC_TIMEOUT_SEC // 60} minutes.\n"
        f"Diagnose with:\n"
        f"  kubectl describe inferenceservice {inference_service_name} -n {kserve_namespace}\n"
        f"  kubectl get pods -n {kserve_namespace}"
    )

print(f"\nInferenceService is Ready! (took {int(time.time() - start)}s)")

[   0s] initializing


[  15s] Configuration "iris-triton-predictor" is waiting for a Revision to become ready.



InferenceService is Ready! (took 30s)


## Retrieve Inference Endpoint URL

KServe sets `status.url` once the Istio/Knative ingress is configured.
If the external URL is not yet propagated, the cell falls back to the cluster-internal DNS name.

In [16]:
status_obj  = ks_client.get(name=inference_service_name, namespace=kserve_namespace)
status_dict = status_obj if isinstance(status_obj, dict) else status_obj.to_dict()

isvc_base_url = status_dict.get("status", {}).get("url", "")

# ── Determine scoring URI ─────────────────────────────────────────────────────
# When running inside the cluster (in-cluster config), the external hostname from
# status.url is typically not DNS-resolvable from within the pod. Instead we use
# the ClusterIP service that KServe/Knative creates for the predictor revision.
# Format: <isvc>-predictor-<revision>.<namespace>.svc.cluster.local

if _using_incluster:
    # Find the ClusterIP service for the predictor revision.
    # KServe creates services labelled serving.knative.dev/service=<isvc>-predictor
    all_svcs = k8s_client.CoreV1Api().list_namespaced_service(
        namespace=kserve_namespace,
        label_selector=f"serving.knative.dev/service={inference_service_name}-predictor",
    ).items
    # Pick the ClusterIP service (not ExternalName) — it has a real cluster IP
    clusterip_svcs = [
        s for s in all_svcs
        if s.spec.type == "ClusterIP" and not s.metadata.name.endswith("-private")
    ]
    if clusterip_svcs:
        svc_name   = clusterip_svcs[0].metadata.name
        svc_host   = f"{svc_name}.{kserve_namespace}.svc.cluster.local"
        scoring_uri = f"http://{svc_host}/v2/models/{triton_model_name}/infer"
        print(f"In-cluster mode: using ClusterIP service '{svc_name}'")
    else:
        # Fallback: standard KServe cluster-internal hostname
        scoring_uri = (
            f"http://{inference_service_name}-predictor.{kserve_namespace}"
            f".svc.cluster.local/v2/models/{triton_model_name}/infer"
        )
        print("In-cluster mode: using standard KServe cluster-internal URL (fallback)")
elif isvc_base_url:
    scoring_uri = f"{isvc_base_url}/v2/models/{triton_model_name}/infer"
    print(f"External URL available: {isvc_base_url}")
else:
    scoring_uri = (
        f"http://{inference_service_name}.{kserve_namespace}"
        f".svc.cluster.local/v2/models/{triton_model_name}/infer"
    )
    print("status.url not set — using cluster-internal URL.")

print(f"\nTriton inference URL : {scoring_uri}")
print()
print("External URL (for access from outside the cluster):")
print(f"  {isvc_base_url}/v2/models/{triton_model_name}/infer" if isvc_base_url else "  (not available)")
print()
print("kubectl port-forward alternative:")
print(
    f"  kubectl port-forward -n {kserve_namespace} "
    f"svc/{inference_service_name}-predictor 8080:80"
)
print(f"  Then use: http://localhost:8080/v2/models/{triton_model_name}/infer")


In-cluster mode: using ClusterIP service 'iris-triton-predictor-00001'

Triton inference URL : http://iris-triton-predictor-00001.unified.svc.cluster.local/v2/models/iris_classifier/infer

External URL (for access from outside the cluster):
  https://iris-triton-predictor-unified.ai-platform.equinor.com/v2/models/iris_classifier/infer

kubectl port-forward alternative:
  kubectl port-forward -n unified svc/iris-triton-predictor 8080:80
  Then use: http://localhost:8080/v2/models/iris_classifier/infer


## Test the KServe Triton Endpoint (KFServing V2 Protocol)

The request format is identical to the AzureML endpoint in `train-on-ai-platform-aks-triton.ipynb`:
- Input tensor `float_input`: 4 × FP32 Iris features
- Output `label`: INT64 predicted class — **0 = setosa, 1 = versicolor, 2 = virginica**
- Output `probabilities`: FP32 `[3]` per-class probabilities

> KServe Triton does not require an auth token by default. If your cluster has auth enabled,
> add `"Authorization": "Bearer <token>"` to the headers.

In [17]:
import requests

headers = {"Content-Type": "application/json"}

payload = {
    "inputs": [
        {
            "name":     "float_input",
            "shape":    [1, n_features],
            "datatype": "FP32",
            "data":     [[5.1, 3.5, 1.4, 0.2]]   # classic setosa sample
        }
    ],
    "outputs": [
        {"name": "label"},
        {"name": "probabilities"}
    ]
}

response = requests.post(scoring_uri, headers=headers, json=payload, verify=False, timeout=30)
print(f"Status  : {response.status_code}")

try:
    parsed = response.json()
    if isinstance(parsed, str):
        parsed = json.loads(parsed)
except json.JSONDecodeError:
    parsed = {"raw": response.text}

print(f"Response:\n{json.dumps(parsed, indent=2)}")

# Interpret results
if response.status_code == 200 and isinstance(parsed, dict) and "outputs" in parsed:
    class_names = ["setosa", "versicolor", "virginica"]
    for out in parsed["outputs"]:
        if out["name"] == "label":
            pred = int(out["data"][0])
            print(f"\nPredicted class   : {pred} ({class_names[pred]})")
        if out["name"] == "probabilities":
            probs = out["data"]
            print("Class probabilities: " +
                  "  ".join(f"{class_names[i]}={probs[i]:.3f}" for i in range(len(class_names))))

# Equivalent curl command (no auth token needed for default KServe)
curl_cmd = (
    f"curl -X POST '{scoring_uri}' "
    f"-H 'Content-Type: application/json' "
    f"-d '{json.dumps(payload)}'"
)
print(f"\nEquivalent curl command:")
print(curl_cmd)

Status  : 200
Response:
{
  "model_name": "iris_classifier",
  "model_version": "1",
  "outputs": [
    {
      "name": "label",
      "datatype": "INT64",
      "shape": [
        1
      ],
      "data": [
        0
      ]
    },
    {
      "name": "probabilities",
      "datatype": "FP32",
      "shape": [
        1,
        3
      ],
      "data": [
        1.0000001192092896,
        0.0,
        0.0
      ]
    }
  ]
}

Predicted class   : 0 (setosa)
Class probabilities: setosa=1.000  versicolor=0.000  virginica=0.000

Equivalent curl command:
curl -X POST 'http://iris-triton-predictor-00001.unified.svc.cluster.local/v2/models/iris_classifier/infer' -H 'Content-Type: application/json' -d '{"inputs": [{"name": "float_input", "shape": [1, 4], "datatype": "FP32", "data": [[5.1, 3.5, 1.4, 0.2]]}], "outputs": [{"name": "label"}, {"name": "probabilities"}]}'


## Cleanup (Optional)

Uncomment to delete the KServe `InferenceService` and the storage credentials secret
when you are done testing. This frees cluster resources and removes stored credentials.

In [18]:
# ── Delete InferenceService ───────────────────────────────────────────────────
# ks_client.delete(name=inference_service_name, namespace=kserve_namespace)
# print(f"InferenceService '{inference_service_name}' deleted.")

# ── Delete storage credentials secret ────────────────────────────────────────
# core_api.delete_namespaced_secret(name=k8s_secret_name, namespace=kserve_namespace)
# print(f"Secret '{k8s_secret_name}' deleted.")

# ── Restore mlpipeline-minio-artifact.secretkey ───────────────────────────────
# Restores the original MinIO password that was overwritten in the deployment cell.
# Uncomment and run after you are done testing the KServe deployment.
# _original = globals().get("_original_minio_secretkey", "")
# if _original:
#     core_api.patch_namespaced_secret(
#         name="mlpipeline-minio-artifact",
#         namespace=kserve_namespace,
#         body={"stringData": {"secretkey": _original}},
#     )
#     print("Restored mlpipeline-minio-artifact.secretkey to original value.")
# else:
#     print("_original_minio_secretkey not set — nothing to restore.")
